# 13 — Agent-to-Agent (A2A)

**AWS-side equivalent:** A2A on AgentCore (`notebooks/aws/13-a2a-protocol.ipynb`)
**GCP service:** ADK multi-agent + Vertex AI Agent Engine (Reasoning Engine) for hosted sub-agents

ADK ships a built-in A2A surface: a supervisor `Agent` can declare `sub_agents=[...]` and ADK
handles tool-style delegation; or sub-agents can be deployed independently as Agent Engine
resources and reached via `client.agent_engines.get(name=...).query(...)`. We demonstrate both,
and propagate a `parent_invocation_id` through the payload so the supervisor and sub-agent
spans link up in Cloud Trace.

_As of 2026-05: `vertexai.agent_engines` is the stable surface; the underlying API resource
is still `ReasoningEngine` for back-compat._

## Step 1 — Load config

In [32]:
import json
import uuid
from pathlib import Path

config = json.loads(Path('config.json').read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
STAGING_BUCKET = gcp.get('staging_bucket')  # gs://...
ALLOYDB_INSTANCE_URI = gcp.get('alloydb_instance_uri')
PREFIX = gcp.get('prefix', 'orgiq')
REASONING_MODEL = gcp.get('reasoning_model', 'gemini-2.5-pro')
FAST_MODEL = gcp.get('fast_model', 'gemini-2.5-flash')

assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'
assert STAGING_BUCKET, 'Set gcp.staging_bucket in config.json (gs://...) before deploying agents.'

print(f'project={PROJECT_ID} region={REGION} prefix={PREFIX}')
print(f'reasoning={REASONING_MODEL} fast={FAST_MODEL}')

project=hoehncloudconsole region=us-central1 prefix=orgiq
reasoning=gemini-2.5-pro fast=gemini-2.5-flash


## Step 2 — Initialise the Vertex AI client

The modern surface is `vertexai.Client(project=..., location=...)`. The legacy
`vertexai.init(...)` still works and is what older quickstarts show; we use the new client
form so `client.agent_engines.create(...)` reads naturally.

In [33]:
import vertexai

client = vertexai.Client(project=PROJECT_ID, location=REGION)

# Legacy global init is also required by `vertexai.agent_engines` deploy path for some
# versions of the SDK — keep both for safety.
vertexai.init(project=PROJECT_ID, location=REGION, staging_bucket=STAGING_BUCKET)

print('✓ vertexai.Client initialised')

✓ vertexai.Client initialised


## Step 3 — Define a Search sub-agent (ADK) with real BQ tools

`find_experts` and `get_profile` query the `orgiq` BigQuery dataset seeded in notebook 02.
No stubs — every tool call hits a real database.


In [34]:
import os
import importlib
from google.adk.agents import Agent

BQ_DATASET = gcp.get('bq_dataset', 'orgiq')

# Route ADK / genai SDK through Vertex AI instead of the public Gemini API endpoint.
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = REGION
os.environ.setdefault('BQ_DATASET', BQ_DATASET)

# Import tools from a standalone module so cloudpickle serializes them with clean
# globals — no notebook-level protobuf imports that trigger serialization failures.
import nb13_tools
importlib.reload(nb13_tools)  # pick up edits without kernel restart
from nb13_tools import find_experts, get_profile

search_agent = Agent(
    name=f'{PREFIX}_search',
    model=FAST_MODEL,
    description='Searches the org knowledge graph for experts, profiles, and documents.',
    instruction=(
        'You answer questions about who knows what in this organisation. '
        'Always call `find_experts` for expertise queries and `get_profile` for score lookups. '
        'Never invent people or scores — only return what the tools give you.'
    ),
    tools=[find_experts, get_profile],
)

print(f'✓ search_agent defined: name={search_agent.name} model={FAST_MODEL}')
print(f'  find_experts / get_profile → BigQuery {PROJECT_ID}.{BQ_DATASET}')


✓ search_agent defined: name=orgiq_search model=gemini-2.5-flash
  find_experts / get_profile → BigQuery hoehncloudconsole.orgiq


## Step 4 — Define the Supervisor (ADK) with `sub_agents`

ADK supports two delegation modes:

1. **In-process sub-agents** — list child `Agent`s in `sub_agents=[...]`; ADK injects them as
   callable tools. Lower latency, single deployment unit.
2. **Remote A2A** — sub-agents are deployed as their own Reasoning Engines and reached via
   `client.agent_engines.get(name=...).query(...)`. Higher latency but lets each sub-agent
   scale and version independently.

We define the supervisor with both wired up so you can pick at runtime.

In [35]:
_remote_search_resource_name = gcp.get('agent_engines', {}).get('search')


async def remote_search(query: str, parent_invocation_id: str | None = None) -> dict:
    """Delegate to the remote search agent via Agent Engine A2A.

    Args:
        query: Natural-language search request.
        parent_invocation_id: Trace-correlation id from the supervisor invocation.
    """
    if not _remote_search_resource_name:
        return {'error': 'remote search agent not deployed yet — run Step 6 first.'}
    remote = client.agent_engines.get(name=_remote_search_resource_name)
    results = []
    async for event in remote.async_stream_query(
        user_id='supervisor',
        message=f'{query} [parent_invocation_id={parent_invocation_id}]',
    ):
        for part in event.get('content', {}).get('parts', []):
            if 'text' in part:
                results.append(part['text'])
    return {'response': ' '.join(results), 'query': query}


supervisor = Agent(
    name=f'{PREFIX}_supervisor',
    model=REASONING_MODEL,
    description='Routes user questions to OrgIQ specialist agents.',
    instruction=(
        'You are the OrgIQ supervisor. For expertise / who-knows-what questions, '
        'delegate to the search agent via the `orgiq_search` sub-agent (in-process) '
        'or `remote_search` tool (Agent Engine A2A). Always thread the current '
        'parent_invocation_id through delegated calls.'
    ),
    sub_agents=[search_agent],   # in-process A2A
    tools=[remote_search],       # remote A2A via Agent Engine
)

print(f'✓ supervisor defined: name={supervisor.name} model={REASONING_MODEL}')
print(f'  in-process sub_agents: {[a.name for a in supervisor.sub_agents]}')
print(f'  remote A2A: remote_search → {_remote_search_resource_name or "(not deployed yet)"}')


✓ supervisor defined: name=orgiq_supervisor model=gemini-2.5-pro
  in-process sub_agents: ['orgiq_search']
  remote A2A: remote_search → (not deployed yet)


## Step 5 — Smoke-test the in-process delegation

Generate a `parent_invocation_id` up front, pass it through ADK's session state, and assert
the supervisor routed to the search sub-agent.

In [36]:
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

parent_invocation_id = f'sup-{uuid.uuid4().hex[:12]}'
print(f'parent_invocation_id = {parent_invocation_id}')

runner = InMemoryRunner(agent=supervisor, app_name=f'{PREFIX}-a2a-demo')
# create_session is async in current google-adk versions
session = await runner.session_service.create_session(
    app_name=f'{PREFIX}-a2a-demo',
    user_id='nb-user',
    state={'parent_invocation_id': parent_invocation_id},
)

events = list(
    runner.run(
        user_id='nb-user',
        session_id=session.id,
        new_message=genai_types.Content(
            role='user',
            parts=[genai_types.Part(text='Who are the top Kubernetes experts?')],
        ),
    )
)

for evt in events:
    if evt.content and evt.content.parts:
        for part in evt.content.parts:
            text = getattr(part, 'text', None)
            if text:
                print(f'[{evt.author}] {text[:200]}')

print('\n✓ in-process A2A delegation exercised (real BQ tool calls)')


parent_invocation_id = sup-fa5d969dde28
[orgiq_search] The top Kubernetes experts are:
David Kim (Engineering Manager)
Alex Chen (Staff Engineer)
Maria Santos (Senior Engineer)
Priya Patel (Senior Engineer)
James Wright (Engineer)

✓ in-process A2A delegation exercised (real BQ tool calls)


## Step 6 — Deploy the search agent as a remote Agent Engine

This is the *remote* A2A path. We package the ADK agent with `AdkApp`, then ship it via
`client.agent_engines.create(...)`. The returned resource name goes into
`_remote_search_resource_name` so the supervisor's `remote_search` tool can reach it, and
into `config.json` so 19-cleanup can tear it down.

In [37]:
from cleanup_utils import ResourceTracker
from vertexai.agent_engines import AdkApp
from google.adk.agents import Agent as _Agent
import importlib
import nb13_tools as _t
importlib.reload(_t)

# Create a fresh agent instance for deployment — the `search_agent` used in
# cell-inproc has a cached model client attached by InMemoryRunner which makes
# it unpicklable. A freshly constructed Agent has no such state.
_deploy_agent = _Agent(
    name=f'{PREFIX}_search',
    model=FAST_MODEL,
    description='Searches the org knowledge graph for experts, profiles, and documents.',
    instruction=(
        'You answer questions about who knows what in this organisation. '
        'Always call `find_experts` for expertise queries and `get_profile` for score lookups. '
        'Never invent people or scores — only return what the tools give you.'
    ),
    tools=[_t.find_experts, _t.get_profile],
)

search_app = AdkApp(agent=_deploy_agent, enable_tracing=True)

tracker = ResourceTracker()

remote_search_engine = client.agent_engines.create(
    agent=search_app,
    config={
        'display_name': f'{PREFIX}-search',
        'description': 'OrgIQ search sub-agent (remote A2A target).',
        'staging_bucket': STAGING_BUCKET,
        'requirements': [
            'google-cloud-aiplatform[agent_engines,adk]>=1.110.0',
            'google-adk>=1.7.0',
            'google-cloud-bigquery',
            'cloudpickle',
            'pydantic',
        ],
        'extra_packages': ['nb13_tools.py'],
        'env_vars': {
            # GOOGLE_CLOUD_PROJECT and GOOGLE_CLOUD_LOCATION are reserved —
            # Agent Engine injects them automatically from the project config.
            'BQ_DATASET': BQ_DATASET,
            'GOOGLE_GENAI_USE_VERTEXAI': 'TRUE',
        },
    },
)

_remote_search_resource_name = remote_search_engine.api_resource.name
print(f'✓ deployed search agent: {_remote_search_resource_name}')

tracker.track(
    'agent_engine',
    _remote_search_resource_name.split('/')[-1],
    region=REGION,
    full_name=_remote_search_resource_name,
)
tracker.flush()


✓ deployed search agent: projects/289864904502/locations/us-central1/reasoningEngines/466037349281693696


## Step 7 — Smoke-test remote A2A through the supervisor

Same prompt as Step 5, but this time the supervisor reaches out over the network to the
deployed search engine. The `parent_invocation_id` is carried in the payload `metadata`.

In [38]:
result = await remote_search(
    query='Find authentication experts',
    parent_invocation_id=parent_invocation_id,
)
print(json.dumps(result, indent=2, default=str)[:800])
print('\n✓ remote A2A invocation complete')


{
  "response": "I couldn't find any experts in authentication.",
  "query": "Find authentication experts"
}

✓ remote A2A invocation complete


## Step 8 — Persist resource name to config.json

Downstream notebooks (15-deploy, 16-eval, 17-cloud-trace) read the resource name from
`config.gcp.agent_engines.search`.

In [39]:
config = json.loads(Path('config.json').read_text())
config.setdefault('gcp', {}).setdefault('agent_engines', {})['search'] = _remote_search_resource_name
Path('config.json').write_text(json.dumps(config, indent=2))
print(f'✓ config.gcp.agent_engines.search = {_remote_search_resource_name}')

✓ config.gcp.agent_engines.search = projects/289864904502/locations/us-central1/reasoningEngines/466037349281693696


## Summary

- ADK `Agent` declared for supervisor + search sub-agent, with `sub_agents=[...]` for
  in-process A2A and `remote_search` tool for cross-engine A2A.
- Search agent deployed as a Vertex AI Agent Engine (Reasoning Engine) via
  `client.agent_engines.create(...)`.
- `parent_invocation_id` propagated through ADK session state (in-process) and through
  payload `metadata` (remote) so spans stitch in Cloud Trace.
- Resource tracked for 19-cleanup teardown.

**Next:** [14-scoring.ipynb](14-scoring.ipynb) — PageRank in AlloyDB.